# Week 5 Practice Tutorial: Parameter Count vs Architecture

The theory tutorial ended by explaining why Adam's default $\alpha = 10^{-3}$ works across very different architectures. This practice tutorial cashes that guarantee: we train four models on Fashion-MNIST with an identical Adam recipe, differing only in architecture, and ask a simple question:

> *Is a bigger model always a better model?*

We build a small CNN and a sequence of MLPs of growing width. Each MLP has **more parameters than the CNN**, yet none will match the CNN's test accuracy. The reason is not tuning; it is the CNN's inductive bias (local connectivity and parameter sharing) which the MLP lacks.

At the end we confirm this by shuffling the pixels of every image: with the CNN's spatial assumption destroyed, its advantage evaporates.


## Setup

Imports, seed, and device selection. Everything runs on Colab CPU in under 10 minutes; a GPU speeds things up but is not required.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## Fashion-MNIST

Fashion-MNIST: 28 x 28 grayscale images of clothing items, 10 classes. Downloaded via `torchvision`. For speed we subset the training set to 20,000 examples; the test set stays at the full 10,000.


In [ ]:
transform = transforms.ToTensor()

full_train = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# Subset training set to 20,000 examples for a Colab-friendly runtime.
subset_indices = torch.randperm(len(full_train))[:20000]
train_data = Subset(full_train, subset_indices)

batch_size = 128
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=1024, shuffle=False)

print(f'Train set: {len(train_data)} examples')
print(f'Test set:  {len(test_data)} examples')
print(f'Classes:   {full_train.classes}')


## What does Fashion-MNIST look like?

Before we start training, take a look at the data. Each image is a 28 x 28 grayscale picture of a clothing item, labelled with one of ten classes.


In [ ]:
# One example from each of the ten classes.
class_examples = {}
for img, label in test_data:
    label = int(label)
    if label not in class_examples:
        class_examples[label] = img
    if len(class_examples) == 10:
        break

fig, axes = plt.subplots(2, 5, figsize=(10, 5.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(class_examples[i].squeeze(), cmap='gray')
    ax.set_title(test_data.classes[i], fontsize=10)
    ax.axis('off')
plt.suptitle('Fashion-MNIST: one example per class', fontsize=12)
plt.tight_layout()
plt.subplots_adjust(hspace=0.4)
plt.show()


## A small CNN

Two conv+pool blocks followed by a small fully-connected head. Around 20 thousand parameters.


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc    = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))   # 28x28 -> 14x14
        x = self.pool(torch.relu(self.conv2(x)))   # 14x14 -> 7x7
        x = x.view(x.size(0), -1)
        return self.fc(x)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

cnn = SmallCNN()
print(f'CNN parameters: {count_params(cnn):,}')


## Two MLPs with more parameters than the CNN

Both MLPs flatten the 28 x 28 image into a 784-dimensional vector, then process it with fully-connected layers. Both have **more parameters than the CNN**.


In [ ]:
class MLPSmall(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 50),
            nn.ReLU(),
            nn.Linear(50, 10),
        )
    def forward(self, x):
        return self.net(x)

class MLPMedium(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 250),
            nn.ReLU(),
            nn.Linear(250, 10),
        )
    def forward(self, x):
        return self.net(x)

mlp_small  = MLPSmall()
mlp_medium = MLPMedium()

print(f'MLP-small  parameters: {count_params(mlp_small):,}')
print(f'MLP-medium parameters: {count_params(mlp_medium):,}')
print(f'CNN        parameters: {count_params(cnn):,}  (for comparison)')


## Training and evaluation

The same recipe for every model: Adam with $\alpha = 10^{-3}$, cross-entropy loss, 7 epochs. Nothing about the training loop changes between models; only the model's architecture does. (Note the choice $\alpha = 10^{-3}$: the theory tutorial's Problem 2 explains why this default is safe across all four architectures below.)


In [ ]:
def train_and_evaluate(model, train_loader, test_loader, epochs=7, lr=1e-3, name=''):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * x.size(0)
        train_loss = running_loss / len(train_loader.dataset)
        print(f'  [{name}] epoch {epoch+1}/{epochs}  train loss: {train_loss:.4f}')

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(dim=1) == y).sum().item()
            total   += y.size(0)
    acc = correct / total
    print(f'  [{name}] test accuracy: {acc:.4f}')
    return acc


Train the CNN and the two MLPs, and collect the test accuracies.


In [ ]:
results = {}

torch.manual_seed(0)
print('Training CNN...')
cnn = SmallCNN()
results['CNN'] = {
    'params': count_params(cnn),
    'acc': train_and_evaluate(cnn, train_loader, test_loader, name='CNN'),
}

torch.manual_seed(0)
print('\nTraining MLP-small...')
mlp_small = MLPSmall()
results['MLP-small'] = {
    'params': count_params(mlp_small),
    'acc': train_and_evaluate(mlp_small, train_loader, test_loader, name='MLP-small'),
}

torch.manual_seed(0)
print('\nTraining MLP-medium...')
mlp_medium = MLPMedium()
results['MLP-medium'] = {
    'params': count_params(mlp_medium),
    'acc': train_and_evaluate(mlp_medium, train_loader, test_loader, name='MLP-medium'),
}


## Comparison

The pattern to notice: MLP-small has roughly twice the CNN's parameter count, MLP-medium roughly ten times, yet neither matches the CNN's test accuracy. The MLPs are not badly broken; they are just less *efficient with their parameters*, because they must learn from data the spatial structure that the CNN gets for free from its architecture.


In [ ]:
print(f"{'Model':<12} {'Params':>10} {'Test acc':>10}")
print('-' * 34)
for name, r in results.items():
    print(f"{name:<12} {r['params']:>10,} {r['acc']:>10.4f}")


> ### <font color="#1b5e20">Exercise 1: An even bigger MLP</font>
>
> Define a two-hidden-layer MLP with the architecture $784 \to 500 \to 500 \to 10$. Call it `MLPBig`.
>
> 1. Instantiate it. How many parameters does it have compared to the CNN?
> 2. Train it with the same recipe (`train_and_evaluate`, default settings) and add the result to the `results` table.
> 3. Reprint the comparison table. What test accuracy did MLP-big achieve? Does it beat the CNN, despite having roughly $30\times$ its parameter count?
>
> *Hint:* use the `MLPMedium` definition as a template.


In [ ]:
class MLPBig(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 500),
            nn.ReLU(),
            nn.Linear(500, 500),
            nn.ReLU(),
            nn.Linear(500, 10),
        )
    def forward(self, x):
        return self.net(x)


# Train MLP-big and record the result.
torch.manual_seed(0)
mlp_big = MLPBig()
print(f'MLP-big parameters: {count_params(mlp_big):,}  '
      f'({count_params(mlp_big) / count_params(cnn):.1f}x the CNN)')
results['MLP-big'] = {
    'params': count_params(mlp_big),
    'acc': train_and_evaluate(mlp_big, train_loader, test_loader, name='MLP-big'),
}

# Reprint comparison.
print(f"\n{'Model':<12} {'Params':>10} {'Test acc':>10}")
print('-' * 34)
for name, r in results.items():
    print(f"{name:<12} {r['params']:>10,} {r['acc']:>10.4f}")


> ### <font color="#1b5e20">Exercise 2 (bonus): Destroy the spatial structure</font>
>
> Why does the CNN win? Two features it has that the MLP does not:
> - **Locality**: each filter looks at a small neighbourhood of pixels.
> - **Parameter sharing**: the same filter is applied at every position.
>
> Both features encode the assumption that *nearby pixels are more related than distant ones*. To isolate this assumption, apply the **same** fixed random permutation to the pixels of every image (both train and test), then retrain a CNN and an MLP on the shuffled data.
>
> The MLP does not care about pixel order (a densely-connected layer is invariant to input reordering, up to a re-labelling of weights), so its accuracy should be roughly unchanged. The CNN's accuracy should drop noticeably: its filters can no longer detect edges or shapes because those depend on adjacency. In fact, on the shuffled data you should find that the CNN loses its edge over the MLP entirely.
>
> **Complete the cell below** to run this experiment. Which model wins on the shuffled data?


In [ ]:
# Fixed random permutation of the 784 pixels.
torch.manual_seed(123)
perm = torch.randperm(784)

class PixelShuffle:
    """Apply a fixed permutation to the pixels of each image."""
    def __init__(self, perm):
        self.perm = perm
    def __call__(self, x):
        return x.view(-1)[self.perm].view(x.shape)

shuffle_transform = transforms.Compose([
    transforms.ToTensor(),
    PixelShuffle(perm),
])

shuffled_train_full = datasets.FashionMNIST(root='./data', train=True,  download=False, transform=shuffle_transform)
shuffled_test       = datasets.FashionMNIST(root='./data', train=False, download=False, transform=shuffle_transform)
shuffled_train      = Subset(shuffled_train_full, subset_indices)

s_train_loader = DataLoader(shuffled_train, batch_size=batch_size, shuffle=True)
s_test_loader  = DataLoader(shuffled_test,  batch_size=1024,       shuffle=False)


# Retrain a fresh CNN on shuffled data.
torch.manual_seed(0)
print('Training CNN on shuffled data...')
cnn_shuf     = SmallCNN()
acc_cnn_shuf = train_and_evaluate(cnn_shuf, s_train_loader, s_test_loader, name='CNN-shuf')

# Retrain a fresh MLP-medium on shuffled data.
torch.manual_seed(0)
print('\nTraining MLP-medium on shuffled data...')
mlp_shuf     = MLPMedium()
acc_mlp_shuf = train_and_evaluate(mlp_shuf, s_train_loader, s_test_loader, name='MLP-medium-shuf')

# Shuffled vs unshuffled comparison.
print(f"\n{'Model':<12} {'Original':>10} {'Shuffled':>10}")
print('-' * 34)
print(f"{'CNN':<12} {results['CNN']['acc']:>10.4f} {acc_cnn_shuf:>10.4f}")
print(f"{'MLP-medium':<12} {results['MLP-medium']['acc']:>10.4f} {acc_mlp_shuf:>10.4f}")


## Reflection

1. **More parameters is not automatically better.** MLP-big has roughly $30\times$ the CNN's parameters and does not beat it on F-MNIST. Raw capacity is only useful when the model can channel that capacity toward the structure that actually generates the data. Without the right inductive bias, extra parameters are largely wasted on redundant re-learning of things a convolution encodes for free.

2. **Locality and parameter sharing, together.** Locality restricts each output to a small neighbourhood of inputs; sharing forces the same detector to be reused at every position. Together they encode the assumption that *the kind of feature that helps identify a shirt-cuff is the same kind that helps identify a shirt-collar*, just at a different location. An MLP has to learn this equivalence from data, which requires seeing the same shape many times in many positions.

3. **The CNN was betting on spatial locality.** Its efficiency on unshuffled images was not general intelligence, it was a bet that nearby pixels are more related than distant ones. Shuffling the pixels removes this structure from the data, so the bet no longer pays off: the CNN drops several percentage points and its lead over the MLP disappears (indeed the MLP may now be ahead). The MLP, which never made the bet, is unaffected.

4. **No: CNNs are the wrong tool for tabular data.** Tabular features (income, number of bedrooms, postcode, etc.) have no spatial relationship: their column order is arbitrary, and there is no notion of neighbouring columns. A CNN's inductive bias is inappropriate here, which is why MLPs and tree-based models (random forests, gradient boosting) dominate tabular ML, while CNNs dominate images.
